# 1. SVM with Scikit-learn

## Concept
In Part 1, we learned the theory behind Support Vector Machines: finding the optimal hyperplane, maximizing the margin, and using the kernel trick for non-linear data. 

In Part 2, we focus purely on **implementation** using Scikit-learn's `SVC` (Support Vector Classification).

### Basic Workflow
When using SVM in practice, the workflow is strict:

1. **Dataset:** Load your data.
2. **Split:** Separate into Training and Test sets *before* doing anything else.
3. **Scale:** Standardize the features (CRITICAL for SVM). Fit scaler on Train, transform Train and Test.
4. **Create SVC:** Instantiate the model with a chosen kernel.
5. **Train:** Fit the model to the scaled training data.
6. **Predict:** Make predictions on the scaled test data.
7. **Evaluate:** Check performance using various metrics.


# 2. Prepare the Dataset

We will use the built-in breast cancer dataset, which is a classic binary classification problem.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# Load dataset
cancer = load_breast_cancer()

# Convert to DataFrame
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target

# Inspect
print("Dataset Shape:", df.shape)
display(df.head())

# Basic EDA: Check class balance
plt.figure(figsize=(4, 3))
sns.countplot(x='target', data=df, palette='bwr')
plt.title('Class Distribution (0: Malignant, 1: Benign)')
plt.show()

# Separate X and y
X = df.drop('target', axis=1)
y = df['target']


# 3. Train-Test Split

We use `train_test_split()` to divide the data.

* `test_size=0.2`: 20% of data for testing, 80% for training.
* `random_state=42`: Ensures reproducibility.
* `stratify=y`: Ensures the train and test sets have the same proportion of malignant/benign cases as the original dataset.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


# 4. Feature Scaling

As discussed in Part 1, **SVM is highly sensitive to the scale of the features** because it relies on computing distances between points. 

If we skip scaling, features with larger ranges (like 'mean area', ranging in the thousands) will dominate features with small ranges (like 'mean smoothness', ranging around 0.1).

### The Correct Workflow:
1. Fit the scaler ONLY on the training data.
2. Transform the training data.
3. Transform the test data using the fitted scaler. (Never fit the scaler on the test data to prevent data leakage!)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Instantiate
scaler = StandardScaler()

# Fit on training data AND transform training data
X_train_scaled = scaler.fit_transform(X_train)

# ONLY transform test data
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame just for visualization
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print("Scaled Data (Notice the values are centered around 0 with std of 1):")
display(X_train_scaled_df.head(3))


# 5. Linear SVM

Let's start with the simplest kernel: the linear kernel. This draws a straight hyperplane.


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# 1. Create model
svc_linear = SVC(kernel="linear", random_state=42)

# 2. Train
svc_linear.fit(X_train_scaled, y_train)

# 3. Predict
y_pred_linear = svc_linear.predict(X_test_scaled)

# 4. Evaluate
print("--- Linear SVM Performance ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_linear):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_linear):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_linear):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_linear):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_linear)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Linear Kernel Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


# 6. Polynomial Kernel

The polynomial kernel maps data into a higher-dimensional space using polynomial functions. We can control the complexity with the `degree` parameter.


In [ ]:
degrees = [2, 3, 4]

print("--- Polynomial Kernel Performance ---")
for d in degrees:
    # Train
    svc_poly = SVC(kernel="poly", degree=d, random_state=42)
    svc_poly.fit(X_train_scaled, y_train)

    # Predict & Evaluate
    acc = accuracy_score(y_test, svc_poly.predict(X_test_scaled))
    print(f"Degree {d} Accuracy: {acc:.4f}")

# Note: Higher degree polynomials often overfit or perform worse than simpler kernels on this dataset.


# 7. RBF Kernel

The **Radial Basis Function (RBF)** is the default and usually the most powerful kernel. 
It measures similarity by distance, placing a Gaussian curve over points. It is excellent for highly complex, nonlinear relationships.


In [ ]:
# 1. Create model (RBF is default, but explicit is better)
svc_rbf = SVC(kernel="rbf", random_state=42)

# 2. Train
svc_rbf.fit(X_train_scaled, y_train)

# 3. Predict
y_pred_rbf = svc_rbf.predict(X_test_scaled)

# 4. Evaluate
print("--- RBF Kernel Performance ---")
print(classification_report(y_test, y_pred_rbf))


# 8. Compare Kernels

Let's do a structured comparison of Linear, Polynomial, RBF, and Sigmoid kernels.


In [ ]:
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
results = []

for k in kernels:
    model = SVC(kernel=k, random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results.append({
        'Kernel': k,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
display(results_df)

# Visualize Comparison
results_df.set_index('Kernel').plot(kind='bar', figsize=(8, 4), colormap='viridis')
plt.title('SVM Kernel Comparison')
plt.ylabel('Score')
plt.ylim(0.8, 1.0)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()


# 9. Experiment with C

`C` is the regularization parameter. 
* Small C: Wide, soft margin. Allows more errors on the training set.
* Large C: Narrow, hard margin. Tries to classify every training point perfectly.


In [ ]:
C_values = [0.01, 0.1, 1, 10, 100]
train_acc = []
test_acc = []

for c in C_values:
    # Using RBF kernel
    model = SVC(kernel='rbf', C=c, random_state=42)
    model.fit(X_train_scaled, y_train)

    train_acc.append(accuracy_score(y_train, model.predict(X_train_scaled)))
    test_acc.append(accuracy_score(y_test, model.predict(X_test_scaled)))

plt.figure(figsize=(7, 4))
plt.plot(C_values, train_acc, label='Training Accuracy', marker='o')
plt.plot(C_values, test_acc, label='Test Accuracy', marker='s')
plt.xscale('log') # Log scale for C
plt.xlabel('C (Log Scale)')
plt.ylabel('Accuracy')
plt.title('Effect of C on RBF SVM Performance')
plt.legend()
plt.grid(True)
plt.show()


**Observation:**
As C increases, the model becomes stricter. Test accuracy improves up to $C=1$ or $C=10$, but at extremely high values ($C=100$), training accuracy reaches 1.0 while test accuracy may drop or plateau—a sign of overfitting.


# 10. Experiment with Gamma

`gamma` determines the influence radius of a single training point in the RBF kernel.
* Small gamma: Broad influence, smooth boundary.
* Large gamma: Tight influence, complex jagged boundary.


In [ ]:
gamma_values = [0.001, 0.01, 0.1, 1, 'scale']
gamma_labels = [str(g) for g in gamma_values]
train_acc_g = []
test_acc_g = []

for g in gamma_values:
    model = SVC(kernel='rbf', gamma=g, random_state=42)
    model.fit(X_train_scaled, y_train)

    train_acc_g.append(accuracy_score(y_train, model.predict(X_train_scaled)))
    test_acc_g.append(accuracy_score(y_test, model.predict(X_test_scaled)))

plt.figure(figsize=(7, 4))
plt.plot(gamma_labels, train_acc_g, label='Training Accuracy', marker='o')
plt.plot(gamma_labels, test_acc_g, label='Test Accuracy', marker='s')
plt.xlabel('Gamma')
plt.ylabel('Accuracy')
plt.title('Effect of Gamma on RBF SVM Performance')
plt.legend()
plt.grid(True)
plt.show()


**Observation:**
Very large gamma (e.g., 1) perfectly memorizes training data (1.0 accuracy) but performs terribly on test data (severe overfitting).


# 11. C and Gamma Together

Because they interact (both control model complexity), they must be tuned together.


In [ ]:
C_vals = [0.1, 1, 10]
G_vals = [0.01, 0.1, 1]
grid_results = []

for c in C_vals:
    for g in G_vals:
        model = SVC(kernel='rbf', C=c, gamma=g, random_state=42)
        model.fit(X_train_scaled, y_train)
        acc = accuracy_score(y_test, model.predict(X_test_scaled))
        grid_results.append({'C': c, 'Gamma': g, 'Test_Accuracy': acc})

grid_df = pd.DataFrame(grid_results)
pivot_df = grid_df.pivot(index='C', columns='Gamma', values='Test_Accuracy')
print("Test Accuracy Grid (C vs Gamma):")
display(pivot_df)


# 12. Confusion Matrix

The confusion matrix breaks down predictions into four categories:
* **TP (True Positive):** Predicted positive, actually positive.
* **TN (True Negative):** Predicted negative, actually negative.
* **FP (False Positive):** Predicted positive, actually negative.
* **FN (False Negative):** Predicted negative, actually positive.


In [ ]:
best_manual_model = SVC(kernel='rbf', C=10, gamma=0.01, random_state=42)
best_manual_model.fit(X_train_scaled, y_train)
y_pred_best = best_manual_model.predict(X_test_scaled)

cm2 = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(4, 3))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['Malignant (0)', 'Benign (1)'], 
            yticklabels=['Malignant (0)', 'Benign (1)'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


# 13. ROC-AUC

* **ROC Curve:** Plots True Positive Rate vs False Positive Rate.
* **AUC:** Area Under the Curve. 1.0 is perfect, 0.5 is guessing.

We use `decision_function()` (distance from hyperplane) to generate the curve.


In [ ]:
from sklearn.metrics import roc_curve, auc

# Get continuous decision values
y_scores = best_manual_model.decision_function(X_test_scaled)

# Calculate FPR, TPR
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()


# 14. Cross Validation

Testing on a single split is noisy. K-Fold Cross Validation splits the **training data** into $K$ folds, evaluating robustly.


In [ ]:
from sklearn.model_selection import cross_val_score

cv_model = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
scores = cross_val_score(cv_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("5-Fold CV Scores:", scores)
print(f"Mean CV Accuracy: {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")


# 15. Hyperparameter Tuning

`GridSearchCV` automates the search for optimal parameters.

> **Important:** The test set must NOT be touched during this process.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'linear']
}

grid = GridSearchCV(SVC(random_state=42), param_grid, refit=True, cv=5, verbose=1)
grid.fit(X_train_scaled, y_train)

print("\nBest Parameters:", grid.best_params_)
print(f"Best CV Score: {grid.best_score_:.4f}")

# Predict on untouched test data:
grid_predictions = grid.predict(X_test_scaled)
print(f"Final Test Accuracy: {accuracy_score(y_test, grid_predictions):.4f}")


# 16. SVM Decision Boundary Visualization

Let's visualize boundaries on synthetic 2D data.


In [ ]:
from sklearn.datasets import make_moons

X_moon, y_moon = make_moons(n_samples=200, noise=0.15, random_state=42)

def plot_boundary(model, X, y, title):
    model.fit(X, y)
    h = .02
    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.6)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors='k')
    plt.title(title)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plot_boundary(SVC(kernel='linear', C=1), X_moon, y_moon, "Linear Kernel")

plt.subplot(1, 3, 2)
plot_boundary(SVC(kernel='rbf', C=1, gamma=1), X_moon, y_moon, "RBF (Good Fit)")

plt.subplot(1, 3, 3)
plot_boundary(SVC(kernel='rbf', C=1, gamma=50), X_moon, y_moon, "RBF (Overfit - High Gamma)")

plt.show()


# 17. Complete End-to-End SVM Project

A complete, production-ready Pipeline workflow.


In [ ]:
from sklearn.pipeline import Pipeline

# 1-3. Load & Split
data = load_breast_cancer()
X_f, X_test_f, y_f, y_test_f = train_test_split(
    pd.DataFrame(data.data, columns=data.feature_names), 
    data.target, test_size=0.2, random_state=42, stratify=data.target)

# 4-5. Pipeline for strict scaling
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42))
])

# 8-10. GridSearch
param_grid_f = {'svm__C': [0.1, 1, 10], 'svm__gamma': [0.01, 0.001], 'svm__kernel': ['rbf', 'linear']}
grid_f = GridSearchCV(pipeline, param_grid_f, cv=5, scoring='accuracy')
grid_f.fit(X_f, y_f)

# 11-12. Evaluate
print("Best Params:", grid_f.best_params_)
y_pred_f = grid_f.predict(X_test_f)
print(f"Final Test Accuracy: {accuracy_score(y_test_f, y_pred_f):.4f}")

# 13. Confusion Matrix
plt.figure(figsize=(3,2))
sns.heatmap(confusion_matrix(y_test_f, y_pred_f), annot=True, fmt='d', cmap='Purples')
plt.title("Final CM")
plt.show()

# 14. ROC-AUC
y_scores_f = grid_f.decision_function(X_test_f)
fpr_f, tpr_f, _ = roc_curve(y_test_f, y_scores_f)
plt.figure(figsize=(3,2))
plt.plot(fpr_f, tpr_f, color='purple', label=f'AUC = {auc(fpr_f, tpr_f):.4f}')
plt.legend()
plt.show()


### Explanation
Using a `Pipeline` inside `GridSearchCV` perfectly scaled the data within cross-validation, avoiding data leakage. The resulting tuned model achieved exceptional generalization on the untouched test set.


# 18. SVM Common Mistakes

* **Forgetting feature scaling:** Destroys distance calculations.
* **Tuning on test data:** Causes data leakage. Use CV instead.
* **Using a huge parameter grid unnecessarily:** Wastes compute. Use log scales.
* **Using inappropriate gamma:** High gamma = extreme overfitting.
* **Using very large C without validation:** Strict margins will overfit outliers.
* **Ignoring class imbalance:** Use `class_weight='balanced'`.
* **Using SVM on extremely large datasets:** $O(n^3)$ scaling is too slow.
* **Data leakage:** Scaling before splitting.
* **Evaluating only accuracy:** Fails on imbalanced data.


# 19. SVM vs Other Algorithms

| Feature | SVM | Logistic Regression | KNN | Decision Tree | Random Forest |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Scaling** | **Crucial** | Crucial | Crucial | Not needed | Not needed |
| **Nonlinearity** | Excellent (Kernels) | Poor | Good | Excellent | Excellent |
| **Interpretability** | Low | High | Low | Very High | Low |
| **Dataset Size** | Small/Medium | Large | Small | Large | Large |
| **Speed** | Slow | Fast | Lazy/Slow | Fast | Moderate |
| **Strengths** | Margin max | Probabilities | Simple | Visual rules | Robust |


# 20. Interview Questions

1. **How do you implement SVM in Python?** `sklearn.svm.SVC`.
2. **Why scale data?** Features with large scales dominate distance metrics.
3. **What scaler is best?** `StandardScaler`.
4. **How to tune parameters?** `GridSearchCV`.
5. **Effect of high C?** Strict, narrow margin. Overfits.
6. **Effect of low C?** Lenient, wide margin. Underfits.
7. **Effect of high gamma?** Tight influence, jagged boundaries, overfits.
8. **Effect of low gamma?** Broad influence, smooth boundaries.
9. **Evaluation metrics?** Accuracy, Precision, Recall, F1, AUC.
10. **Why use a Pipeline?** Prevent data leakage during scaling in CV.
11. **Can SVM output probabilities?** Yes, by setting `probability=True`.
12. **Linear vs RBF?** Linear for high-dim/simple data, RBF for complex nonlinear data.
13. **What is `decision_function`?** Distance from hyperplane.
14. **How to handle imbalance?** `class_weight='balanced'`.
15. **Why cross-validate?** Single splits can give lucky/unlucky results.
16. **Training complexity?** $O(n^3)$.
17. **Alternative for large data?** `SGDClassifier` or `LinearSVC`.
18. **What does stratify do?** Keeps class ratio equal in train/test splits.
19. **Can it use categorical data?** Must be numerical/One-Hot encoded first.
20. **Is test data scaled with train data?** Test data is transformed using the scaler fitted *only* on train data.


# 21. Quick Revision

### SVM Practical Workflow
Data → Split → Scale (Fit Train, Transform Both) → Select Kernel → Train SVC → Tune C/Gamma → CV → Final Test → Evaluate

### Cheat Sheet
* **Model:** `SVC(kernel='rbf', C=1.0)`
* **Scaling:** `StandardScaler().fit_transform(X_train)`
* **Tuning:** `GridSearchCV(model, param_grid)`
* **Evaluation:** `classification_report()`, `roc_curve()`


# 22. Practice Problems

1. Train linear SVM.
2. Train RBF SVM.
3. Compare kernels.
4. Compare C values.
5. Compare gamma values.
6. Create decision boundary.
7. Perform cross-validation.
8. Use GridSearchCV.
9. Compare SVM with Logistic Regression.
10. Build an SVM classification pipeline.

---

## Next Notebook
`10_kmeans_clustering.ipynb`

We are now moving from Supervised Learning to Unsupervised Learning with **K-Means Clustering**!
